# UrbanNav Odaiba — validazione mappa satelliti vs RINEX

Confronta i conteggi della **mappa area** (`mean_n_visible`, `mean_n_los`, `mean_n_nlos`) con i satelliti tracciati nel RINEX UrbanNav lungo la stessa traiettoria.

**Runtime:** GPU (T4). Compila le estensioni CUDA `_bvh`.

**Output** in `/content/odaiba_work/results/`:
- mappa area CSV/HTML
- confronto epoca-per-epoca CSV + summary JSON
- scatter, timeline PNG, mappa residui HTML

In [ ]:
# 1) Verifica GPU
!nvidia-smi

In [ ]:
# 2) Clone repo (sostituisci URL con il tuo fork se necessario)
import os
REPO = "/content/gnss_gpu"
if not os.path.isdir(REPO):
    !git clone https://github.com/YOUR_USER/gnss_gpu.git {REPO}
%cd {REPO}

In [ ]:
# 3) Dipendenze Python + build CUDA (T4 = sm_75)
!pip install -q numpy matplotlib folium branca pyproj rasterio requests scipy
!apt-get -qq install -y cmake
!mkdir -p build
%cd build
!cmake .. -DCMAKE_CUDA_ARCHITECTURES=75
!make -j$(nproc)
%cd ..

In [ ]:
# 4) Test import estensioni
import os, sys
os.environ["PYTHONPATH"] = "python:build"
sys.path[:0] = ["python", "build"]
import gnss_gpu._bvh
print("_bvh OK")

In [ ]:
# 5) Pipeline completa: fetch → mesh → mappa → validazione
import os
os.environ["PYTHONPATH"] = "python:build"
!python experiments/run_odaiba_map_validation_colab.py \
  --work-dir /content/odaiba_work \
  --phase all \
  --epoch-step 5 \
  --instantaneous-geometry

In [ ]:
# 6) Mostra risultati
from IPython.display import Image, display, JSON
import json
from pathlib import Path

R = Path("/content/odaiba_work/results")
summary = json.loads((R / "odaiba_map_vs_rinex_summary.json").read_text())
display(JSON(summary))
display(Image(filename=str(R / "odaiba_map_vs_rinex_scatter.png")))
display(Image(filename=str(R / "odaiba_map_vs_rinex_timeline.png")))

In [ ]:
# 7) Scarica zip risultati su Drive/local
!zip -r /content/odaiba_validation_results.zip /content/odaiba_work/results
from google.colab import files
files.download("/content/odaiba_validation_results.zip")

## Fasi singole (debug)

```bash
python experiments/run_odaiba_map_validation_colab.py --phase fetch --work-dir /content/odaiba_work
python experiments/run_odaiba_map_validation_colab.py --phase mesh --work-dir /content/odaiba_work --skip-fetch
python experiments/run_odaiba_map_validation_colab.py --phase map --work-dir /content/odaiba_work --skip-fetch
python experiments/run_odaiba_map_validation_colab.py --phase validate --work-dir /content/odaiba_work --skip-fetch
```

Se hai già i dati su Drive, monta Drive e usa `--skip-fetch` con `--work-dir` puntato alla cartella dati.